In [1]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams 

PROJECT_ROOT = Path.cwd() 

if PROJECT_ROOT.name == "notebooks": 
    PROJECT_ROOT = PROJECT_ROOT.parent 

sys.path.insert(0, str(PROJECT_ROOT)) 

from src.embedder import Embedder 

/Users/aob/AOB/CS/Projects/clip-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 18:04:44.046000 98656 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
embeddings_path = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "abo_image_embeddings.npy"
)

metadata_path = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "abo_image_metadata.csv"
)

image_embeddings = np.load(embeddings_path)
metadata = pd.read_csv(metadata_path)

print("Embedding shape:", image_embeddings.shape)
print("Metadata rows:", len(metadata))

Embedding shape: (1000, 512)
Metadata rows: 1000


In [5]:
if image_embeddings.shape[0] != len(metadata): 
    raise ValueError("The number of embeddings does not match metadata rows.") 

vector_size = image_embeddings.shape[1] 

print("Vector size:", vector_size) 
print("Embedding and metadata rows are aligned.")

Vector size: 512
Embedding and metadata rows are aligned.


In [8]:
# Create an in-memmory Qdrant client 

client = QdrantClient(":memory:") 

print("Qdrant in-memory client created.")

Qdrant in-memory client created.


#### The `client` is the connnection to Qdrant. 

#### With `":memory:"`: 

*   no server is required; 
*   no local databse file is created; 
*   all data lives in memory; 
*   data disappears when the client or kernel stops. 

### **Create a collection**: A `collection` is similar to a table in a traidiotnal databse. It stores vectors with the same dimensionality and distance metric 

In [10]:
collection_name = "abo_products" 

client.create_collection(
    collection_name=collection_name, 
    vectors_config=VectorParams(
        size=vector_size, 
        distance=Distance.COSINE, 
    ), 
)

print(f"Created collection: {collection_name}")

Created collection: abo_products


In [12]:
collection_info = client.get_collection(collection_name) 
print(collection_info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=512, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, pr

#### **Qdrant point**: 

A Qdrant point contains three conceptual pieces: 
```
PointStruct(
    id=some ID, 
    vector=the 512-dimensional embedding, 
    payload= metadata dictionary, 
)
```

In [13]:
# Upload all 1,000 points 
from qdrant_client.models import Batch 

In [16]:
ids = list(range(len(metadata))) 

payloads = [
    {
        "item_id": row["item_id"], 
        "image_id": row["image_id"], 
        "product_type": row["product_type"], 
        "title": row["title"], 
        "image_path": row["image_path"], 
    }
    for _, row in metadata.iterrows()
]

client.upload_collection(
    collection_name=collection_name, 
    vectors=image_embeddings.tolist(), 
    payload=payloads, 
    ids=ids, 
)

print(f"Uploaded {len(ids)} points.")

Uploaded 1000 points.


In [18]:
count_results = client.count(
    collection_name=collection_name, 
    exact=True, 
)

print("Stored points:", count_results.count)

Stored points: 1000


In [19]:
# Load the text embedder 

embedder = Embedder() 

print("Device:", embedder.device)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 67801.19it/s]


Device: mps


In [20]:
# Search Qdrant 
query = "a black handbag" 
query_embedding = embedder.embed_text(query) 

search_result = client.query_points(
    collection_name=collection_name, 
    query=query_embedding.tolist(), 
    limit=5, 
    with_payload=True, 
)

print(search_result) 

points=[ScoredPoint(id=941, version=0, score=0.30643938277970784, payload={'item_id': 'B081HMLLBR', 'image_id': '71HQLq+AUzL', 'product_type': 'HANDBAG', 'title': "Nia & Nicole Women's Handbag (Navy Blue)", 'image_path': 'data/ABO/abo-images-small/images/small/0f/0fa0e93d.jpg'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=37, version=0, score=0.3048613876371785, payload={'item_id': 'B000R5H79K', 'image_id': '41pxABUoPoL', 'product_type': 'CELLULAR_PHONE_CASE', 'title': 'Rivet The Wrap Fashion Case for Mobile Phones', 'image_path': 'data/ABO/abo-images-small/images/small/5c/5c4c18c9.jpg'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=952, version=0, score=0.3016345408254819, payload={'item_id': 'B07WRJFTRK', 'image_id': '71Ps6Eena3L', 'product_type': 'HANDBAG', 'title': "Flavia Women's Handbag (Black)", 'image_path': 'data/ABO/abo-images-small/images/small/c4/c4270338.jpg'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=912, version=

In [22]:
for rank, point in enumerate(search_result.points, start=1): 
    print(f"\nRank {rank}")
    print("Score", point.score)
    print("Product type:", point.payload["product_type"]) 
    print("Title", point.payload["title"]) 
    print("Image path:", point.payload["image_path"]) 


Rank 1
Score 0.30643938277970784
Product type: HANDBAG
Title Nia & Nicole Women's Handbag (Navy Blue)
Image path: data/ABO/abo-images-small/images/small/0f/0fa0e93d.jpg

Rank 2
Score 0.3048613876371785
Product type: CELLULAR_PHONE_CASE
Title Rivet The Wrap Fashion Case for Mobile Phones
Image path: data/ABO/abo-images-small/images/small/5c/5c4c18c9.jpg

Rank 3
Score 0.3016345408254819
Product type: HANDBAG
Title Flavia Women's Handbag (Black)
Image path: data/ABO/abo-images-small/images/small/c4/c4270338.jpg

Rank 4
Score 0.296398481475287
Product type: HANDBAG
Title The Lovely bolsa Co. - Bolera de nailon con asa superior a rayas en contraste con pompones de piel sintética para mujer, Bolero de nailon con pompón de pelo a rayas de color
Image path: data/ABO/abo-images-small/images/small/ad/add5ebec.jpg

Rank 5
Score 0.29393578259735254
Product type: HANDBAG
Title WindTook Small Crossbody Bag for Women Shoulder Bags Ladies Purses Clutch Bag for Girls Party Outdoor(WT-020 Black)
Image 

In [23]:
numpy_scores = image_embeddings @ query_embedding 
numpy_indices = np.argsort(numpy_scores)[::-1][:5] 

print("NumPy results:")
for rank, index in enumerate(numpy_indices, start=1):
    print(
        rank,
        metadata.iloc[index]["product_type"],
        float(numpy_scores[index]),
    )

NumPy results:
1 HANDBAG 0.3064393997192383
2 CELLULAR_PHONE_CASE 0.3048613667488098
3 HANDBAG 0.3016345798969269
4 HANDBAG 0.29639849066734314
5 HANDBAG 0.2939358353614807


In [24]:
print("\nQdrant results:")
for rank, point in enumerate(search_result.points, start=1):
    print(
        rank,
        point.payload["product_type"],
        point.score,
    )


Qdrant results:
1 HANDBAG 0.30643938277970784
2 CELLULAR_PHONE_CASE 0.3048613876371785
3 HANDBAG 0.3016345408254819
4 HANDBAG 0.296398481475287
5 HANDBAG 0.29393578259735254


In [25]:
queries = [
    "a black handbag",
    "comfortable running shoes",
    "a blue phone case",
    "modern furniture",
    "home decor",
]

In [26]:
for query in queries: 
    query_embedding = embedder.embed_text(query) 

    numpy_scores = image_embeddings @ query_embedding 
    numpy_indices = np.argsort(numpy_scores)[::-1][:5] 

    qdrant_result = client.query_points(
        collection_name=collection_name, 
        query=query_embedding.tolist(), 
        limit=5, 
        with_payload=True, 
    )

    numpy_item_ids = [
        metadata.iloc[index]["item_id"] 
        for index in numpy_indices
    ]

    qdrant_item_ids = [
        point.payload["item_id"] 
        for point in qdrant_result.points
    ]

    print(f"\nQuery: {query}") 
    print("NumPy IDs: ", numpy_item_ids) 
    print("Qdrant IDs: ", qdrant_item_ids) 
    print("Exact match:", numpy_item_ids == qdrant_item_ids) 


Query: a black handbag
NumPy IDs:  ['B081HMLLBR', 'B000R5H79K', 'B07WRJFTRK', 'B06X17B2N4', 'B08F9ZWWXP']
Qdrant IDs:  ['B081HMLLBR', 'B000R5H79K', 'B07WRJFTRK', 'B06X17B2N4', 'B08F9ZWWXP']
Exact match: True

Query: comfortable running shoes
NumPy IDs:  ['B083PGXKB9', 'B07QJ32BFN', 'B082BY6R1W', 'B083SFKPV4', 'B0797F2TB8']
Qdrant IDs:  ['B083PGXKB9', 'B07QJ32BFN', 'B082BY6R1W', 'B083SFKPV4', 'B0797F2TB8']
Exact match: True

Query: a blue phone case
NumPy IDs:  ['B082DNQR4W', 'B08562DD6Z', 'B079J6N6PK', 'B07VXRRB3J', 'B004I8WD7S']
Qdrant IDs:  ['B082DNQR4W', 'B08562DD6Z', 'B079J6N6PK', 'B07VXRRB3J', 'B004I8WD7S']
Exact match: True

Query: modern furniture
NumPy IDs:  ['B07HPDV36G', 'B001BQKT3K', 'B07JVP2C7M', 'B001RTS5B4', 'B07J55B89H']
Qdrant IDs:  ['B07HPDV36G', 'B001BQKT3K', 'B07JVP2C7M', 'B001RTS5B4', 'B07J55B89H']
Exact match: True

Query: home decor
NumPy IDs:  ['B076VC9T9V', 'B08FHMSF6C', 'B08FH56JHR', 'B085FBPWXT', 'B08FHLLWQC']
Qdrant IDs:  ['B076VC9T9V', 'B08FHMSF6C', 'B08FH5